# Sesión 2: Recuperación

En esta sesión construimos perfiles de usuario a partir de los juegos consumidos y los usamos como consultas del sistema de recuperación de información.

In [1]:
import ast
import json
import os
import re

import faiss
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

c:\Users\Willy\Desktop\Master\Sistemas de Recuperación de Información y de Recomendación\Practica\Rag\rag-steam-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Rutas

PROJECT_ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
OUTPUTS_ROOT = os.path.join(PROJECT_ROOT, "outputs")
SESSION1_OUTPUTS_DIR = os.path.join(OUTPUTS_ROOT, "session1_indexing")
OUTPUTS_DIR = os.path.join(OUTPUTS_ROOT, "session2_retrieval")

USER_ITEMS_PATH = os.path.join(DATA_DIR, "steam_user.json")
GAMES_DATASET_PATH = os.path.join(SESSION1_OUTPUTS_DIR, "games_dataset.csv")
FAISS_INDEX_PATH = os.path.join(SESSION1_OUTPUTS_DIR, "faiss.index")

for path in [USER_ITEMS_PATH, GAMES_DATASET_PATH, FAISS_INDEX_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró el archivo {path}. Ejecuta antes la sesión 1 desde la carpeta notebooks/."
        )

os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [3]:
# Resultados de la sesion 1

games_df = pd.read_csv(GAMES_DATASET_PATH)
games_df["item_id"] = games_df["item_id"].astype(str)

item_id_to_name = dict(zip(games_df["item_id"], games_df["item_name"]))
item_id_to_document = dict(zip(games_df["item_id"], games_df["document"]))

index = faiss.read_index(FAISS_INDEX_PATH)

In [4]:
games_df.head()

,item_id,item_name,num_reviews,document
0,251570,7 Days to Die,20,7 days to die this game is awesome its like mi...
1,250420,8BitMMO,20,8bitmmo this game is so much fun i love buildi...
2,113400,APB Reloaded,20,apb reloaded i do hate the fact that i lagged ...
3,346110,ARK: Survival Evolved,20,ark survival evolved so i play ark on and off ...
4,224540,Ace of Spades,20,ace of spades meh awesome great game yeah a pr...


## Lectura y limpieza de texto

In [6]:
def read_json_lines(path, max_lines=None):
    """Lee un archivo pseudo-JSONL"""
    records = []

    with open(path, "r", encoding="utf-8") as f:
        iterator = tqdm(f, desc=f"Leyendo {os.path.basename(path)}", unit="line")

        for i, line in enumerate(iterator):
            if max_lines is not None and i >= max_lines:
                break

            line = line.strip()
            if not line:
                continue

            try:
                records.append(ast.literal_eval(line))
            except (SyntaxError, ValueError):
                continue

    return records

def clean_text(text):
    """Limpia texto eliminando HTML, saltos de línea y caracteres raros"""
    if text is None:
        return ""

    text = BeautifulSoup(str(text), "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"_+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Construcción de perfiles de usuario

In [ ]:
# El objetivo es transformar el historial de juegos de cada usuario en un perfil textual.
# Para ello se combinan los nombres de los juegos con parte de sus documentos asociados
# (reseñas procesadas en la sesión 1), de forma que después pueda generarse un embedding del perfil
# y usarlo como consulta para recuperar juegos similares.

def build_user_profiles(
    user_items_path,
    valid_item_ids,
    item_id_to_document,
    min_items=5,
    max_items_per_user=15,
    max_profile_docs=5,
    max_users=20,
):
    """Construye perfiles de usuario con nombres y documentos de juegos usados."""
    records = read_json_lines(user_items_path)
    rows = []

    for user_record in tqdm(records, desc="Construyendo perfiles de usuario"):
        user_id = str(user_record.get("user_id", "")).strip()
        items = user_record.get("items", [])

        if not user_id or not isinstance(items, list):
            continue

        valid_items = []

        for item_data in items:
            if not isinstance(item_data, dict):
                continue

            item_id = str(item_data.get("item_id", "")).strip()
            item_name = str(item_data.get("item_name", "")).strip()
            playtime_value = item_data.get("playtime_forever", 0)

            try:
                playtime_forever = float(playtime_value)
            except (TypeError, ValueError):
                continue

            if not item_id or not item_name:
                continue

            if item_id not in valid_item_ids or playtime_forever <= 0:
                continue

            valid_items.append(
                {
                    "item_id": item_id,
                    "item_name": item_name,
                    "playtime_forever": playtime_forever,
                }
            )

        if not valid_items:
            continue

        valid_items = sorted(
            valid_items,
            key=lambda item: (-item["playtime_forever"], item["item_name"]),
        )[:max_items_per_user]

        if len(valid_items) < min_items:
            continue

        consumed_item_ids = [item["item_id"] for item in valid_items]
        consumed_item_names = [item["item_name"] for item in valid_items]

        profile_docs = [
            item_id_to_document[item_id][:1000]
            for item_id in consumed_item_ids[:max_profile_docs]
            if item_id in item_id_to_document
        ]

        profile_text = clean_text(
            " ".join(consumed_item_names) + " " + " ".join(profile_docs)
        )

        if not profile_text:
            continue

        rows.append(
            {
                "user_id": user_id,
                "num_items": len(consumed_item_ids),
                "profile_text": profile_text,
                "consumed_item_ids": consumed_item_ids,
                "consumed_item_names": consumed_item_names,
            }
        )

        if len(rows) >= max_users:
            break

    columns = [
        "user_id",
        "num_items",
        "profile_text",
        "consumed_item_ids",
        "consumed_item_names",
    ]
    return pd.DataFrame(rows, columns=columns)

In [8]:
valid_item_ids = set(games_df["item_id"].astype(str))

user_profiles_df = build_user_profiles(
    USER_ITEMS_PATH,
    valid_item_ids,
    item_id_to_document,
    min_items=5,
    max_items_per_user=15,
    max_profile_docs=5,
    max_users=20,
)

if user_profiles_df.empty:
    raise ValueError(
        "No se han generado perfiles de usuario."
    )

Leyendo steam_user.json: 88310line [01:24, 1041.60line/s]
Construyendo perfiles de usuario:   0%|          | 21/88310 [00:00<01:00, 1448.02it/s]


In [9]:
user_profiles_df.head()

,user_id,num_items,profile_text,consumed_item_ids,consumed_item_names
0,76561197970982479,15,counter strike global offensive rising storm r...,"[730, 35450, 8930, 1250, 232090, 24960, 24980,...","[Counter-Strike: Global Offensive, Rising Stor..."
1,js41637,15,the elder scrolls v skyrim terraria saints row...,"[72850, 105600, 55230, 620, 238010, 28050, 242...","[The Elder Scrolls V: Skyrim, Terraria, Saints..."
2,evcentric,15,gnomoria fallout new vegas borderlands fallout...,"[224500, 22380, 8980, 377160, 49520, 28050, 10...","[Gnomoria, Fallout: New Vegas, Borderlands, Fa..."
3,Riot-Punch,15,grand theft auto iv street fighter iv the witc...,"[12210, 21660, 292030, 72850, 12750, 8870, 620...","[Grand Theft Auto IV, Street Fighter IV, The W..."
4,doctr,15,counter strike global offensive borderlands 2 ...,"[730, 49520, 311210, 550, 218620, 22380, 37716...","[Counter-Strike: Global Offensive, Borderlands..."


In [10]:
user_profiles_df["num_items"].describe()

count    20.000000
mean     14.750000
std       1.118034
min      10.000000
25%      15.000000
50%      15.000000
75%      15.000000
max      15.000000
Name: num_items, dtype: float64

## Ejemplo de perfil de usuario generado

In [11]:

example_profile = user_profiles_df.iloc[0]

print("Usuario:", example_profile["user_id"])
print("Juegos usados en el perfil:", example_profile["num_items"])
print("\nJuegos más representativos:")
for i, game in enumerate(example_profile["consumed_item_names"][:10], start=1):
    print(f"{i}. {game}")

print("\nFragmento del perfil textual:")
print(example_profile["profile_text"][:700] + "...")

Usuario: 76561197970982479
Juegos usados en el perfil: 15

Juegos más representativos:
1. Counter-Strike: Global Offensive
2. Rising Storm/Red Orchestra 2 Multiplayer
3. Sid Meier's Civilization V
4. Killing Floor
5. Killing Floor 2
6. Battlefield: Bad Company 2
7. Mass Effect 2
8. Day of Defeat: Source
9. Dragon Age: Origins
10. Plants vs. Zombies: Game of the Year

Fragmento del perfil textual:
counter strike global offensive rising storm red orchestra 2 multiplayer sid meier s civilization v killing floor killing floor 2 battlefield bad company 2 mass effect 2 day of defeat source dragon age origins plants vs zombies game of the year xcom enemy unknown just cause 2 borderlands insurgency final fantasy vii counter strike global offensive hold shift to win hold ctrl to lose zika do baile best game in the bloody world good game it was very fun i really do recommend anyone who has played any cs or cod like games to give this game a go it may not be like cod but it is still as much fun m

In [12]:
# Guardar perfiles de usuario

user_profiles_path = os.path.join(OUTPUTS_DIR, "user_profiles.csv")

user_profiles_to_save = user_profiles_df.copy()
user_profiles_to_save["consumed_item_ids"] = user_profiles_to_save["consumed_item_ids"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)
user_profiles_to_save["consumed_item_names"] = user_profiles_to_save["consumed_item_names"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

user_profiles_to_save.to_csv(user_profiles_path, index=False, encoding="utf-8")
print(f"Perfiles guardados en: {user_profiles_path}")

Perfiles guardados en: ..\outputs\session2_retrieval\user_profiles.csv


## Generacion de embeddings

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

documents = user_profiles_df["profile_text"].tolist()
user_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
)

user_embeddings = np.asarray(user_embeddings, dtype=np.float32)
faiss.normalize_L2(user_embeddings)
user_embeddings.shape

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]


(20, 384)

In [14]:
# Recuperacion de juegos similares a los perfiles de usuario FAISS

final_k = 10
search_k = min(50, index.ntotal) # Por si algun juego ya lo ha jugado el usuario

scores, indices = index.search(user_embeddings, search_k)

retrieval_rows = []

for user_position, user_row in user_profiles_df.reset_index(drop=True).iterrows():
    consumed_item_ids = {str(item_id) for item_id in user_row["consumed_item_ids"]}
    consumed_item_names = user_row["consumed_item_names"]
    user_id = user_row["user_id"]
    profile_text = user_row["profile_text"]

    rank = 1
    for score, idx in zip(scores[user_position], indices[user_position]):
        if idx < 0:
            continue

        game_row = games_df.iloc[int(idx)]
        item_id = str(game_row["item_id"])
        item_name = item_id_to_name.get(item_id, game_row["item_name"])

        if item_id in consumed_item_ids:
            continue

        retrieval_rows.append(
            {
                "user_id": user_id,
                "rank": rank,
                "item_id": item_id,
                "item_name": item_name,
                "score": float(score),
                "profile_text": profile_text,
                "consumed_item_names": consumed_item_names,
            }
        )

        rank += 1
        if rank > final_k:
            break

retrieval_results = pd.DataFrame(
    retrieval_rows,
    columns=[
        "user_id",
        "rank",
        "item_id",
        "item_name",
        "score",
        "profile_text",
        "consumed_item_names",
    ],
)

if retrieval_results.empty:
    raise ValueError("No se han generado recomendaciones. Revisa los perfiles o la recuperación.")

retrieval_results.head()

,user_id,rank,item_id,item_name,score,profile_text,consumed_item_names
0,76561197970982479,1,42700,Call of Duty: Black Ops,0.697285,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
1,76561197970982479,2,10090,Call of Duty: World at War,0.692121,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
2,76561197970982479,3,33930,Arma 2: Operation Arrowhead,0.685877,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
3,76561197970982479,4,10,Counter-Strike,0.672896,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
4,76561197970982479,5,202970,Call of Duty: Black Ops II,0.669079,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."


In [15]:
# Guardar resultados de recuperación

retrieval_results_path = os.path.join(OUTPUTS_DIR, "retrieval_results.csv")

retrieval_results_to_save = retrieval_results.copy()
retrieval_results_to_save["consumed_item_names"] = retrieval_results_to_save["consumed_item_names"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

retrieval_results_to_save.to_csv(retrieval_results_path, index=False, encoding="utf-8")
print(f"Resultados guardados en: {retrieval_results_path}")

Resultados guardados en: ..\outputs\session2_retrieval\retrieval_results.csv


## Conclusión

En esta sesión se han creado perfiles de usuario usando sus juegos más jugados. Después, esos perfiles se han convertido en vectores y se han usado para buscar en FAISS juegos parecidos. El resultado es una lista de candidatos recomendados para cada usuario, que se usará en la sesión 3 para que el LLM los reordene y los explique.